# Chapter 7 — End-to-End AgentCore Evaluation

This notebook walks through the **complete evaluation lifecycle** in a single place:

1. **Deploy** a travel assistant agent to AgentCore Runtime
2. **Create** a custom evaluator (LLM-as-judge)
3. **Invoke** the agent with a set of test prompts
4. **Evaluate** the session with built-in and custom metrics
5. **Inspect** and summarise the results

### The Agent

A travel assistant that can answer questions about:
- ✈️  Flights between cities
- 🏨  Hotel recommendations by budget
- 🌤️  Weather forecasts

It should **refuse** to answer questions outside the travel domain — we will verify this with our custom evaluator.

### Tutorial Details

| Information        | Details                                              |
|:-------------------|:-----------------------------------------------------|
| Agent framework    | Strands Agents SDK                                   |
| LLM model          | Anthropic Claude Haiku 3.5 (via Amazon Bedrock)      |
| Deployment         | AgentCore Runtime (CodeBuild, no local Docker needed)|
| Evaluation type    | On-demand (built-in + custom)                        |
| SDK used           | AgentCore Starter Toolkit                            |
| Complexity         | Beginner                                             |

### Prerequisites
- Python 3.10+
- AWS credentials configured (`aws configure` or environment variables)
- IAM permissions: `bedrock-agentcore:*`, `bedrock-agentcore-control:*`, `ecr:*`, `iam:CreateRole`, `codebuild:*`, `s3:*`, `logs:*`

---
## Step 0 — Install dependencies

In [3]:
%pip install -r requirements.txt -q
%pip install --upgrade bedrock-agentcore-starter-toolkit -q

/home/agent/.venvs/ai-agents-ch6-7/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.
/home/agent/.venvs/ai-agents-ch6-7/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


---
## Step 1 — Setup: imports and AWS session

In [4]:
import json
import time
import uuid

import boto3
from boto3.session import Session
from IPython.display import Markdown, display

from bedrock_agentcore_starter_toolkit import Evaluation, Runtime

boto_session = Session()
region = boto_session.region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]

print(f"Region  : {region}")
print(f"Account : {account_id}")

Region  : us-east-1
Account : 685394474162


---
## Step 2 — Deploy the travel agent to AgentCore Runtime

We use the AgentCore Starter Toolkit `Runtime` class to:
- Configure the agent (entrypoint, IAM role, ECR repo)
- Build the container image via AWS CodeBuild (no local Docker required)
- Deploy to AgentCore Runtime

> ⏱️ First-time deployment takes ~10 minutes (CodeBuild + ECR push). Subsequent updates are faster.

In [5]:
runtime = Runtime()

runtime.configure(
    entrypoint="agent_app.py",
    agent_name="ch7_travel_agent",
    requirements_file="requirements.txt",
    region=region,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    idle_timeout=120,
)

launch_result = runtime.launch()
print(f"\nDeployment started: {launch_result}")

Entrypoint parsed: file=/c/Data/OSS/AI-Agents-on-AWS/chapter 7/agent-evaluation/agent_app.py, bedrock_agentcore_name=agent_app
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: ch7_travel_agent
Memory disabled
Network mode: PUBLIC


⚠️ Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64', so local builds
won't work.
Please use default launch command which will do a remote cross-platform build using code build.For deployment other
options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

📄 Generated Dockerfile: /c/Data/OSS/AI-Agents-on-AWS/chapter 7/agent-evaluation/Dockerfile

Generated .dockerignore: /c/Data/OSS/AI-Agents-on-AWS/chapter 7/agent-evaluation/.dockerignore
Setting 'ch7_travel_agent' as default agent
Bedrock AgentCore configured: /c/Data/OSS/AI-Agents-on-AWS/chapter 7/agent-evaluation/.bedrock_agentcore.yaml
🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'ch7_travel_agent' to account 685394474162 (us-east-1)
Generated image tag: 20260904-064633-477
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: ch7_travel_agent


Repository doesn't exist, creating new ECR repository: bedrock-agentcore-ch7_travel_agent


ECR repository available: 685394474162.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-ch7_travel_agent
Getting or creating execution role for agent: ch7_travel_agent
Using AWS region: us-east-1, account ID: 685394474162
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-6727f255fe
Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-6727f255fe
Starting execution role creation process for agent: ch7_travel_agent
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-us-east-1-6727f255fe
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-6727f255fe
✓ Role created: arn:aws:iam::685394474162:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-6727f255fe
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-ch7_travel_agent
Role creation complete and ready for use with Bedrock AgentCore
Execution role available: arn:aws:iam::685394474162:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-6727f255fe
Preparing CodeBuild project and upl


Deployment started: mode='codebuild' tag='bedrock_agentcore-ch7_travel_agent:None' env_vars=None port=None runtime=None ecr_uri='685394474162.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-ch7_travel_agent:20260904-064633-477' agent_id='ch7_travel_agent-wVbg6F91e5' agent_arn='arn:aws:bedrock-agentcore:us-east-1:685394474162:runtime/ch7_travel_agent-wVbg6F91e5' codebuild_id='bedrock-agentcore-ch7_travel_agent-builder:179ec866-8136-4110-9775-197c580ee205' build_output=None


In [6]:
def wait_for_ready(runtime, name="Agent", poll_seconds=15):
    """Poll until the agent runtime reaches a terminal status."""
    terminal = {"READY", "CREATE_FAILED", "UPDATE_FAILED", "DELETE_FAILED"}
    while True:
        status = runtime.status().endpoint["status"]
        print(f"  {name} status: {status}")
        if status in terminal:
            return status
        time.sleep(poll_seconds)

status = wait_for_ready(runtime, name="Travel Agent")
assert status == "READY", f"Deployment failed with status: {status}"
print(f"\n✅ Agent is READY")
print(f"   Agent ID  : {launch_result.agent_id}")
print(f"   Agent ARN : {launch_result.agent_arn}")

Retrieved Bedrock AgentCore status for: ch7_travel_agent


  Travel Agent status: READY

✅ Agent is READY
   Agent ID  : ch7_travel_agent-wVbg6F91e5
   Agent ARN : arn:aws:bedrock-agentcore:us-east-1:685394474162:runtime/ch7_travel_agent-wVbg6F91e5


---
## Step 3 — Create a custom evaluator

We create a **travel quality** evaluator that scores responses on a 5-point scale and penalises the agent for answering out-of-scope questions.

The evaluator config is stored in `travel_quality_metric.json`.

In [7]:
eval_client = Evaluation(region=region)

# List available built-in evaluators
print("Built-in evaluators available:")
available = eval_client.list_evaluators()
for e in available.get("evaluators", []):
    print(f"  - {e['evaluatorId']}")

/home/agent/.venvs/ai-agents-ch6-7/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets"
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Built-in evaluators available:


Built-in Evaluators (18)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID                                ┃ Name                          ┃ Level      ┃ Description                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Builtin.Coherence                 │ Builtin.Coherence             │ TRACE      │ Response Quality Metric.       │
│                                   │                               │            │ Evaluates whether the response │
│                                   │                               │            │ is logically structured and    │
│                                   │                               │            │ coherent                       │
│ Builtin.Conciseness               │ Builtin.Conciseness           │ TRACE      │ Response Quality Metric.       │
│                                   │                               │            │ Evaluates whether the response │
│                                   │                               │            │ is appropriately brief without │
│                                   │                               │            │ missing key information        │
│ Builtin.Correctness               │ Builtin.Correctness           │ TRACE      │ Response Quality Metric.       │
│                                   │                               │            │ Evaluates whether the          │
│                                   │                               │            │ information in the agent's     │
│                                   │                               │            │ response is factually accurate │
│ Builtin.Faithfulness              │ Builtin.Faithfulness          │ TRACE      │ Response Quality Metric.       │
│                                   │                               │            │ Evaluates whether information  │
│                                   │                               │            │ in the response is supported   │
│                                   │                               │            │ by provided context/sources    │
│ Builtin.GoalSuccessRate           │ Builtin.GoalSuccessRate       │ SESSION    │ Task Completion Metric.        │
│                                   │                               │            │ Evaluates whether the          │
│                                   │                               │            │ conversation successfully      │
│                                   │                               │            │ meets the user's goals         │
│ Builtin.Harmfulness               │ Builtin.Harmfulness           │ TRACE      │ Safety Metric. Evaluates       │
│                                   │                               │            │ whether the response contains  │
│                                   │                               │            │ harmful content                │
│ Builtin.Helpfulness               │ Builtin.Helpfulness           │ TRACE      │ Response Quality Metric.       │
│                                   │                               │            │ Evaluates from user's          │
│                                   │                               │            │ perspective how useful and     │
│                                   │                               │            │ valuable the agent's response  │
│                                   │                               │            │ is                             │
│ Builtin.InstructionFollowing      │ Builtin.InstructionFollowing  │ TRACE      │ Response Quality Metric.       │
│                                   │                               │            │ Measures how well the agent    │
│                                   │                               │            │ follows the provided system    │
│                                   │                   

Custom Evaluators (13)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID                                           ┃ Name                     ┃ Level      ┃ Description              ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ThirdParty.DeepEval.Bias                     │ ThirdParty.DeepEval.Bias │ TRACE      │ Safety Metric            │
│                                              │                          │            │ (DeepEval). Evaluates    │
│                                              │                          │            │ whether the agent's      │
│                                              │                          │            │ response shows gender,   │
│                                              │                          │            │ political, racial, or    │
│                                              │                          │            │ geographical bias.       │
│ ThirdParty.DeepEval.Toxicity                 │ ThirdParty.DeepEval.Tox… │ TRACE      │ Safety Metric            │
│                                              │                          │            │ (DeepEval). Evaluates    │
│                                              │                          │            │ whether the agent's      │
│                                              │                          │            │ response contains        │
│                                              │                          │            │ attacks, mockery, hate,  │
│                                              │                          │            │ or threats.              │
│ ThirdParty.DeepEval.PIILeakage               │ ThirdParty.DeepEval.PII… │ TRACE      │ Safety Metric            │
│                                              │                          │            │ (DeepEval). Evaluates    │
│                                              │                          │            │ whether the agent's      │
│                                              │                          │            │ response exposes         │
│                                              │                          │            │ personal information.    │
│ ThirdParty.DeepEval.Summarization            │ ThirdParty.DeepEval.Sum… │ TRACE      │ Response Quality Metric  │
│                                              │                          │            │ (DeepEval). Evaluates    │
│                                              │                          │            │ whether the agent's      │
│                                              │                          │            │ summary is faithful and  │
│                                              │                          │            │ comprehensive.           │
│ ThirdParty.DeepEval.TaskCompletion           │ ThirdParty.DeepEval.Tas… │ TRACE      │ Task Completion Metric   │
│                                              │                          │            │ (DeepEval). Evaluates    │
│                                              │                          │            │ whether the agent        │
│                                              │                          │            │ accomplished the user's  │
│                                              │                          │            │ goal.                    │
│ ThirdParty.DeepEval.ConversationCompleteness │ ThirdParty.DeepEval.Con… │ SESSION    │ Conversational Metric    │
│                                              │                          │            │ (DeepEval). Evaluates    │
│                                              │                          │            │ whether all user         │
│                                              │                          │            │ requests across the      │
│                                              │        

Total: 31 (18 builtin, 13 custom)

  - Builtin.Correctness
  - Builtin.Faithfulness
  - Builtin.Helpfulness
  - Builtin.ResponseRelevance
  - Builtin.Conciseness
  - Builtin.Coherence
  - Builtin.InstructionFollowing
  - Builtin.Refusal
  - Builtin.GoalSuccessRate
  - Builtin.ToolSelectionAccuracy
  - Builtin.ToolParameterAccuracy
  - Builtin.Harmfulness
  - Builtin.Stereotyping
  - Builtin.TrajectoryExactOrderMatch
  - Builtin.TrajectoryInOrderMatch
  - Builtin.TrajectoryAnyOrderMatch
  - Builtin.SkillSelectionAccuracy
  - Builtin.SkillInstructionFollowing
  - ThirdParty.DeepEval.Bias
  - ThirdParty.DeepEval.Toxicity
  - ThirdParty.DeepEval.PIILeakage
  - ThirdParty.DeepEval.Summarization
  - ThirdParty.DeepEval.TaskCompletion
  - ThirdParty.DeepEval.ConversationCompleteness
  - ThirdParty.DeepEval.KnowledgeRetention
  - ThirdParty.DeepEval.TurnRelevancy
  - ThirdParty.DeepEval.GoalAccuracy
  - ThirdParty.DeepEval.ToolUse
  - ThirdParty.AutoEval.Security
  - ThirdParty.AutoEval.Humor
  - ThirdParty.AutoEval.Possible


In [8]:
with open("travel_quality_metric.json") as f:
    eval_config = json.load(f)

custom_evaluator = eval_client.create_evaluator(
    name="travel_response_quality",
    level="TRACE",
    description="Evaluates travel assistant response quality and scope adherence",
    config=eval_config,
)

evaluator_id = custom_evaluator["evaluatorId"]
print(f"✅ Custom evaluator created: {evaluator_id}")

✓ Evaluator created successfully!

ID: travel_response_quality-XN2a387S5r

ARN: arn:aws:bedrock-agentcore:us-east-1:685394474162:evaluator/travel_response_quality-XN2a387S5r

Use: eval_client.run(evaluators=['travel_response_quality-XN2a387S5r'])

✅ Custom evaluator created: travel_response_quality-XN2a387S5r


---
## Step 4 — Invoke the agent with test prompts

We send a mix of prompts:
- ✅ In-scope travel questions (should score high)
- ❌ Out-of-scope question (should score low with our custom evaluator)

All prompts share the same `session_id` so we can evaluate the full session.

In [9]:
session_id = str(uuid.uuid4())
print(f"Session ID: {session_id}\n")

test_prompts = [
    "What flights are available from New York to London?",
    "Can you recommend a mid-range hotel in Paris?",
    "What is the weather like in London right now?",
    "Can you help me write a Python script to sort a list?",  # out-of-scope
]

responses = []
for prompt in test_prompts:
    print(f"📤 Prompt: {prompt}")
    response = runtime.invoke(
        payload={"prompt": prompt},
        session_id=session_id,
    )
    print(f"🤖 Response: {response}\n")
    responses.append({"prompt": prompt, "response": response})

print(f"✅ Invoked {len(test_prompts)} prompts in session {session_id}")

Session ID: 260f8114-a29f-4036-96a0-7b659accac91

📤 Prompt: What flights are available from New York to London?
🤖 Response: {'ResponseMetadata': {'RequestId': 'a1cb3ea0-df65-42aa-8687-5b7c47be036f', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Fri, 04 Sep 2026 07:14:05 GMT', 'content-type': 'application/json', 'transfer-encoding': 'chunked', 'connection': 'keep-alive', 'x-amzn-requestid': 'a1cb3ea0-df65-42aa-8687-5b7c47be036f', 'x-amzn-bedrock-agentcore-runtime-session-id': '260f8114-a29f-4036-96a0-7b659accac91'}, 'RetryAttempts': 0}, 'runtimeSessionId': '260f8114-a29f-4036-96a0-7b659accac91', 'contentType': 'application/json', 'statusCode': 200, 'response': ["Great! Here's the flight information from New York to London:\n\n**Flight AA100**\n- **Departure:** JFK at 10:00 PM\n- **Arrival:** LHR (London Heathrow) at 10:00 AM (next day)\n- **Duration:** 7 hours\n- **Price:** $650\n\nThis is a direct flight option available for your journey from New York to London. Would you like help w

---
## Step 5 — Wait for traces to appear in AgentCore Observability

AgentCore Runtime emits OpenTelemetry traces to CloudWatch. We need to wait ~2 minutes for them to be ingested before running evaluations.

In [ ]:
TRACE_INGESTION_WAIT = 180  # seconds (3 minutes — safe default)

print(f"Waiting {TRACE_INGESTION_WAIT}s for traces to be ingested into AgentCore Observability...")
for remaining in range(TRACE_INGESTION_WAIT, 0, -10):
    print(f"  {remaining}s remaining...")
    time.sleep(10)
print("✅ Ready to evaluate")

---
## Step 6 — Run evaluations

We run three evaluation passes on the same session:

| Pass | Evaluator | Level | What it measures |
|------|-----------|-------|------------------|
| A | `Builtin.GoalSuccessRate` | Session | Did the user achieve their goals? |
| B | `Builtin.Correctness` + `Builtin.Helpfulness` | Trace | Per-turn accuracy and usefulness |
| C | `travel_response_quality` (custom) | Trace | Scope adherence + quality |

### Pass A — Session-level: Goal Success Rate

In [ ]:
goal_results = eval_client.run(
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=["Builtin.GoalSuccessRate"],
)

print("=== Goal Success Rate (session level) ===")
for result in goal_results.results:
    display(Markdown(f"""
**Result:** {result.label} ({result.value})

**Explanation:** {result.explanation}

**Token usage:** {result.token_usage}
"""))

### Pass B — Trace-level: Correctness and Helpfulness

In [ ]:
quality_results = eval_client.run(
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=["Builtin.Correctness", "Builtin.Helpfulness"],
)

print("=== Correctness & Helpfulness (trace level) ===")
for result in quality_results.results:
    display(Markdown(f"""
**Metric:** {result.evaluator_name}
**Result:** {result.label} ({result.value})
**Explanation:** {result.explanation}
"""))
    print("---")

### Pass C — Custom evaluator: Travel Quality & Scope Adherence

This is where we expect the out-of-scope Python question to score **Very Poor (0.0)**.

In [ ]:
custom_results = eval_client.run(
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=[evaluator_id],
)

print("=== Travel Quality — Custom Evaluator (trace level) ===")
for result in custom_results.results:
    emoji = "✅" if result.value >= 0.75 else ("⚠️" if result.value >= 0.5 else "❌")
    display(Markdown(f"""
{emoji} **Result:** {result.label} ({result.value})

**Explanation:** {result.explanation}
"""))
    print("---")

---
## Step 7 — Summary

Aggregate all results into a simple score table.

In [ ]:
all_results = (
    list(goal_results.results)
    + list(quality_results.results)
    + list(custom_results.results)
)

print(f"{'Evaluator':<35} {'Label':<15} {'Score':>6}")
print("-" * 60)
for r in all_results:
    name = getattr(r, "evaluator_name", evaluator_id)
    label = getattr(r, "label", "N/A")
    value = getattr(r, "value", 0)
    bar = "█" * int(value * 10)
    print(f"{name:<35} {label:<15} {value:>5.2f}  {bar}")

numeric_values = [getattr(r, "value", 0) for r in all_results if getattr(r, "value", None) is not None]
if numeric_values:
    avg = sum(numeric_values) / len(numeric_values)
    print("-" * 60)
    print(f"{'Average score':<35} {'':15} {avg:>5.2f}")

---
## Step 8 — Save results to file

In [ ]:
import os

os.makedirs("eval_output", exist_ok=True)

save_results = eval_client.run(
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=["Builtin.Correctness", evaluator_id],
    output="eval_output/chapter7_results.json",
)

print("✅ Results saved to eval_output/chapter7_results.json")

---
## (Optional) Cleanup

Uncomment and run the cell below to delete the agent runtime and free up resources.

In [11]:
runtime.destroy()
print("Agent runtime deleted")

🗑️ Destroying Bedrock AgentCore resources
Starting destroy operation for agent: ch7_travel_agent (dry_run=False, delete_ecr_repo=False)
DEFAULT endpoint will be automatically deleted with agent
Deleted AgentCore agent: arn:aws:bedrock-agentcore:us-east-1:685394474162:runtime/ch7_travel_agent-wVbg6F91e5
Checking ECR repository: bedrock-agentcore-ch7_travel_agent in region: us-east-1
Deleted 1 ECR images from bedrock-agentcore-ch7_travel_agent
Deleted CodeBuild project: bedrock-agentcore-ch7_travel_agent-builder
Deleted S3 artifact: ch7_travel_agent/deployment.zip
Deleted S3 artifact: ch7_travel_agent/source.zip
Deleted inline policy CodeBuildExecutionPolicy from role AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-6727f255fe
Deleted CodeBuild IAM role: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-6727f255fe
Deleted IAM role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-6727f255fe
Removed agent configuration: ch7_travel_agent
Cleared default agent (no agents remaining)
Removed configuration f

Agent runtime deleted


---
## What you just did

In a single notebook you:

1. **Deployed** a Strands agent to AgentCore Runtime using CodeBuild (no Docker needed)
2. **Created** a custom LLM-as-judge evaluator with a 5-point scale
3. **Invoked** the agent with in-scope and out-of-scope prompts
4. **Evaluated** the session with built-in metrics (GoalSuccessRate, Correctness, Helpfulness) and your custom metric
5. **Verified** that the custom evaluator correctly penalised the out-of-scope response